# SmartDs 2D Slice Plot (Starter)

This notebook reads an already-2D BATSRUS slice file and plots a **flat-shaded triangulated field** on the native 2D mesh.

It reuses the library (`SmartDs`) and the file's native mesh connectivity (`sds.corners`) and does **not** resample.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from starwinds_analysis.smart_ds import SmartDs

plt.rcParams['figure.dpi'] = 120

## Choose a 2D File

By default this looks for a `z=0*.plt` slice in `sample_data/`.

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'sample_data').exists():
    # If launched from somewhere else, fall back to this repo root.
    REPO_ROOT = Path('/Users/dagfev/Documents/starwinds/starwinds-analysis')

sample_data = REPO_ROOT / 'sample_data'
z0_candidates = sorted(sample_data.glob('z=0*.plt'))

if not z0_candidates:
    available = sorted(p.name for p in sample_data.glob('*.plt'))
    raise FileNotFoundError(
        'No z=0 .plt file found in sample_data. Available .plt files: ' + ', '.join(available)
    )

DATA_FILE = z0_candidates[0]
print('Using:', DATA_FILE)

sds = SmartDs.from_file(str(DATA_FILE))
print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))

In [ ]:
# Inspect available fields
list(sds.variables)[:40]

## Auto-Detect Slice Coordinates and Choose a Field

The notebook inspects `X [R]`, `Y [R]`, `Z [R]` and picks the two coordinates that vary across the 2D slice.

In [ ]:
coord_candidates = ['X [R]', 'Y [R]', 'Z [R]']
coord_arrays = {}
coord_spans = {}

for name in coord_candidates:
    arr = np.asarray(sds.variable(name), dtype=float)
    coord_arrays[name] = arr
    finite = arr[np.isfinite(arr)]
    coord_spans[name] = float(np.ptp(finite)) if finite.size else 0.0

max_span = max(coord_spans.values()) if coord_spans else 0.0
tol = max(1e-12, 1e-9 * max_span)
varying = [name for name in coord_candidates if coord_spans[name] > tol]
constant = [name for name in coord_candidates if coord_spans[name] <= tol]

if len(varying) != 2:
    # Fallback: choose the two largest spans, even if the thresholding is awkward.
    varying = sorted(coord_candidates, key=lambda n: coord_spans[n], reverse=True)[:2]
    constant = [name for name in coord_candidates if name not in varying]

X_FIELD, Y_FIELD = varying
COLOR_FIELD = 'Rho [g/cm^3]'  # change as needed

print('Coordinate spans:', coord_spans)
print('Detected in-plane coordinates:', X_FIELD, Y_FIELD)
print('Detected (near) constant coordinate(s):', constant)

x = np.asarray(coord_arrays[X_FIELD], dtype=float)
y = np.asarray(coord_arrays[Y_FIELD], dtype=float)
c = np.asarray(sds.variable(COLOR_FIELD), dtype=float)

if not (np.isfinite(x).all() and np.isfinite(y).all()):
    raise ValueError('Non-finite in-plane coordinates found; this starter expects a clean 2D slice mesh.')

finite_c = np.isfinite(c)
print('Finite field values:', int(finite_c.sum()), '/', c.size)

In [ ]:
import matplotlib.tri as mtri

corners = np.asarray(sds.corners, dtype=int)
if corners.ndim != 2 or corners.shape[1] not in (3, 4):
    raise ValueError(f'Expected triangular or quad connectivity, got shape {corners.shape}')

if corners.shape[1] == 4:
    triangles = np.vstack([
        corners[:, [0, 1, 2]],
        corners[:, [0, 2, 3]],
    ])
else:
    triangles = corners

tris = mtri.Triangulation(x, y, triangles)
print('Cells (quads):', corners.shape[0])
print('Triangles   :', triangles.shape[0])

In [ ]:
# Flat-shaded triangulated plot on the native 2D mesh (no resampling)
fig, ax = plt.subplots(figsize=(7, 6))

c_plot = np.ma.masked_invalid(c)
finite = np.isfinite(c)
use_log = finite.any() and np.nanmin(c[finite]) > 0
norm = LogNorm() if use_log else None

img = ax.tripcolor(tris, c_plot, shading='flat', cmap='viridis', norm=norm)
ax.triplot(tris, color='k', linewidth=0.15, alpha=0.15)

ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title(COLOR_FIELD)
fig.colorbar(img, ax=ax, label=COLOR_FIELD)
plt.show()

## Alfvén Surface (2D Contour)

This uses the BATSRUS recipe graph to compute the Alfvén Mach number `M_A` on demand and draws the `M_A = 1` contour as a 2D Alfvén-surface proxy in the slice plane.

In [ ]:
# 2D Alfvén-surface proxy: M_A = 1 contour on the native slice mesh
sds.add_batsrus_graph()
ma = np.asarray(sds.variable('M_A [none]'), dtype=float)
ma_plot = np.ma.masked_invalid(ma)
ma_plot = np.ma.masked_less_equal(ma_plot, 0.0)  # LogNorm requires positive values

fig, ax = plt.subplots(figsize=(7, 6))
cmap = plt.get_cmap('cividis').copy()
cmap.set_under('#440154')  # outside color for M_A < 1e-2
cmap.set_over('#fde725')   # outside color for M_A > 1e2
norm = LogNorm(vmin=1e-2, vmax=1e2)
img = ax.tripcolor(tris, ma_plot, shading='flat', cmap=cmap, norm=norm)

# Draw the Alfvén surface in 2D as the M_A = 1 contour.
valid = np.isfinite(ma)
if valid.any() and (np.nanmin(ma[valid]) <= 1.0 <= np.nanmax(ma[valid])):
    cs = ax.tricontour(tris, ma, levels=[1.0], colors='crimson', linewidths=1.5)
    try:
        cs.collections[0].set_label('M_A = 1')
        ax.legend(loc='best')
    except Exception:
        pass
else:
    print('M_A = 1 contour not present within this slice/value range.')

ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title('Alfvén Mach number M_A [none] (log scale 1e-2 to 1e2)')
fig.colorbar(img, ax=ax, label='M_A [none]', extend='both')
plt.show()